# Deploy the trained SmolVLA policy (always-on, command driven)

Run on your **laptop** from the repo root. The simulation keeps running; you send a tool name and the arm picks it, puts it in the green tray, returns home, and waits.

Commands are received over a tiny local UDP socket, so you can send them from:
- the companion notebook **`send_command.ipynb`** (open it in a second tab - buttons or `send('hammer')`)
- a terminal: `python send_command.py hammer`
- later, your teammate's gesture program (it just sends the same text to `127.0.0.1:5005`)

Accepted commands: `wrench`, `screwdriver`, `hammer`, `pliers`, `reset` (reshuffle tools), `quit`.
(Any sentence that contains a tool name works too, e.g. `pick the hammer`.)

In [4]:
import queue, socket, threading
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.configs.policies import PreTrainedConfig
from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from mujoco_env.y_env_tools import SimpleEnvTools, TOOLS, PROMPT

In [5]:
ROOT      = './demo_data_garage'            # only meta/stats.json is needed from the dataset
REPO_NAME = 'garage_pnp'
CKPT      = './ckpt/smolvla_garage/checkpoints/004000/pretrained_model'   # <- the step you downloaded from Colab
XML       = './asset/example_scene_tools.xml'
PORT      = 5005

MAX_RUN_TICKS = 600     # give up on a command after 30 s of sim time (20 Hz)
MIN_RUN_TICKS = 30      # do not test for success right at the start
HOME_TICKS    = 60      # 3 s scripted return to the home pose between commands

### Load the policy  (CPU works but is slow: expect a pause every 10 steps)

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

meta = LeRobotDatasetMetadata(REPO_NAME, root=ROOT)
cfg = PreTrainedConfig.from_pretrained(CKPT)
cfg.device = device
policy = SmolVLAPolicy.from_pretrained(CKPT, config=cfg, dataset_stats=meta.stats).to(device).eval()
print("chunk_size", cfg.chunk_size, " n_action_steps", cfg.n_action_steps)
to_tensor = transforms.ToTensor()

device: cpu
Reducing the number of VLM layers to 16 ...
Loading weights from local directory
chunk_size 10  n_action_steps 10


### Command listener  (runs in the background; re-running this cell is safe)

In [7]:
cmd_q = queue.Queue()
try:
    sock.close()                       # re-run safe: free the port from a previous run
except Exception:
    pass
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
sock.bind(('127.0.0.1', PORT))

def listener(s):
    while True:
        try:
            data, _ = s.recvfrom(1024)
        except OSError:                # socket closed
            return
        cmd_q.put(data.decode().strip())

threading.Thread(target=listener, args=(sock,), daemon=True).start()
print(f"listening on 127.0.0.1:{PORT}")

def parse(text):
    t = text.strip().lower()
    if t in ('reset', 'quit', 'exit'):
        return t
    for name in TOOLS:
        if name in t:
            return name
    return None

listening on 127.0.0.1:5005


In [8]:
def obs_for(env):
    rgb, wrist = env.grab_image()
    f = lambda a: to_tensor(Image.fromarray(a).resize((256, 256))).unsqueeze(0).to(device)
    return {'observation.state': torch.tensor(env.get_joint_state()).unsqueeze(0).to(device),   # 8-D
            'observation.image': f(rgb),
            'observation.wrist_image': f(wrist),
            'task': [env.instruction]}

PnPEnv = SimpleEnvTools(XML, action_type='joint_angle', seed=0)


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[39] n_qvel:[35] n_qacc:[35] n_ctrl:[11]
 integrator:[IMPLICITFAST]

n_body:[24]
 [0/24] [world] mass:[0.00]kg
 [1/24] [front_object_table] mass:[1.00]kg
 [2/24] [camera] mass:[0.00]kg
 [3/24] [camera2] mass:[0.00]kg
 [4/24] [camera3] mass:[0.00]kg
 [5/24] [slider_carriage] mass:[12.00]kg
 [6/24] [link1] mass:[2.06]kg
 [7/24] [link2] mass:[3.68]kg
 [8/24] [link3] mass:[2.39]kg
 [9/24] [link4] mass:[1.40]kg
 [10/24] [link5] mass:[1.40]kg
 [11/24] [link6] mass:[0.65]kg
 [12/24] [camera_center] mass:[0.00]kg
 [13/24] [tcp_link] mass:[0.32]kg
 [14/24] [rh_p12_rn_r1] mass:[0.07]kg
 [15/24] [rh_p12_rn_r2] mass:[0.02]kg
 [16/24] [rh_p12_rn_l1] mass:[0.07]kg
 [17/24] [rh_p12_rn_l2] mass:[0.02]kg
 [18/24] [body_tray_src] mass:[4.34]kg
 [19/24] [body_tray_dst] mass:[4.34]kg
 [20/24] [body_obj_wrench] mass:[0.09]kg
 [21/24] [body_obj_screwdriver] mass:[0.07]kg
 [22/24] [body_

### Run the always-on loop
After this cell starts, switch to **send_command.ipynb** (or a terminal) and send a tool name.
Stop with the command `quit`, or by closing the viewer window.

In [9]:
mode, tool, t, home_from = 'idle', None, 0, None
policy.reset()
print("ready. send one of:", TOOLS)

while PnPEnv.env.is_viewer_alive():
    PnPEnv.step_env()
    if not PnPEnv.env.loop_every(HZ=20):
        continue

    # ---------------- idle: look for a new command ----------------
    if mode == 'idle' and not cmd_q.empty():
        cmd = parse(cmd_q.get())
        if cmd in ('quit', 'exit'):
            break
        elif cmd == 'reset':
            PnPEnv.reset(); PnPEnv.set_instruction('')
            print("scene reset")
        elif cmd is None:
            print("unknown command; use one of", TOOLS)
        elif PnPEnv.tool_in_tray(cmd, 'dst'):
            print(f"{cmd} is already in the destination tray")
        else:
            tool, t, mode = cmd, 0, 'run'
            PnPEnv.set_instruction(PROMPT.format(cmd))
            policy.reset()                              # clear the previous action chunk
            print("->", PnPEnv.instruction)

    # ---------------- run: the policy drives the arm ----------------
    if mode == 'run':
        with torch.no_grad():
            action = policy.select_action(obs_for(PnPEnv))[0, :8].cpu().numpy()   # 6 joints, slide, gripper
        PnPEnv.step(action)
        t += 1
        if t >= MIN_RUN_TICKS and PnPEnv.check_success(tool):
            print(f"success: {tool}")
            mode, t, home_from = 'home', 0, PnPEnv.get_action_vec()
        elif t >= MAX_RUN_TICKS:
            print(f"timeout on {tool} (send 'reset' to reshuffle if it is stuck)")
            mode, t, home_from = 'home', 0, PnPEnv.get_action_vec()

    # ---------------- home: scripted return so every command starts from the training start pose ----------------
    elif mode == 'home':
        t += 1
        a = min(1.0, t / HOME_TICKS)
        vec = (1 - a) * home_from + a * PnPEnv.home_action_vec()
        vec[7] = 0.0                                    # open gripper
        PnPEnv.step(vec)
        if t >= HOME_TICKS + 10:
            mode = 'idle'
            PnPEnv.set_instruction('')
            print("idle - waiting for a command")

    if mode != 'run':
        PnPEnv.grab_image()                             # keep the overlay cameras fresh
    PnPEnv.render()

ready. send one of: ['wrench', 'screwdriver', 'hammer', 'pliers']
-> pick up the hammer and place it in the tray
timeout on hammer (send 'reset' to reshuffle if it is stuck)
idle - waiting for a command
-> pick up the pliers and place it in the tray
timeout on pliers (send 'reset' to reshuffle if it is stuck)
idle - waiting for a command
-> pick up the hammer and place it in the tray


In [10]:
PnPEnv.env.close_viewer()
sock.close()